# TKCE — Deep-dive: training dynamics (standalone)

Trains **TKCE-joint** on ONE balanced binary dataset (`eye_movements`, 50/50) with a
**much deeper** Siamese encoder + TabResNet, **600 epochs**, a **slow learning rate**,
and **no early stopping** — then plots **loss** and **AUC/accuracy per epoch** for both
the **training** and **validation** sets.

**This notebook is independent of the main suite** — you can run it in a separate
session while the main run continues.

**Setup:** `Runtime → Change runtime type → A100 (or H100 / L4) GPU`. This experiment
is pure neural-network training, so a strong GPU really does speed it up.
Expected time: ~10–40 min depending on GPU.

## 1. Config — edit only if you want different settings

In [ ]:
REPO_URL   = "https://github.com/sushanedulloo/TKCE.git"
DRIVE_BASE = "/content/drive/MyDrive/tkce_results"

# ---- experiment settings ("deep + long + slow", multiple shuffles) ----
TASK      = 361070          # eye_movements: binary, perfectly balanced
SEEDS     = "0,1,2,3,4"     # 5 shuffles = 5 different random splits
EPOCHS    = 600             # 500+ as requested
LR        = 1e-4            # slow learning rate
ENC_WIDTH = 512             # Siamese encoder width
ENC_DEPTH = 6               # Siamese encoder depth (deep)
EMB       = 128             # embedding size
N_BLOCKS  = 8               # TabResNet residual blocks (deep)
D         = 256
D_HIDDEN  = 512
HEAD      = "tabresnet"     # or "mlp"

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
FIG = DRIVE_BASE + "/figures"
AN  = DRIVE_BASE + "/analysis"
LOG = DRIVE_BASE + "/deep_joint_run.log"
for d in (DRIVE_BASE, FIG, AN): os.makedirs(d, exist_ok=True)
print('Results will be saved to:', FIG, 'and', AN)

## 3. Get the code + install

In [ ]:
%cd /content
!rm -rf tkce_deep
!git clone $REPO_URL tkce_deep
%cd tkce_deep
!pip install -q -r requirements-tkce.txt
print('Ready')

## 4. GPU check

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
!nvidia-smi -L

## 5. Run the deep-dive experiment
Progress prints live. It trains the full number of epochs (no early stopping) so the
complete trend is visible.

In [ ]:
!python -u run_deep_joint.py --task $TASK --seeds $SEEDS --epochs $EPOCHS --lr $LR --enc-width $ENC_WIDTH --enc-depth $ENC_DEPTH --embedding-dim $EMB --d $D --d-hidden $D_HIDDEN --n-blocks $N_BLOCKS --head $HEAD --device auto --out "$FIG" --csv "$AN" 2>&1 | tee "$LOG"

## 6. Show the resulting figure

In [ ]:
from IPython.display import Image, display
import glob
pngs = sorted(glob.glob(FIG + '/deep_joint_*.png'))
print(pngs)
for p in pngs: display(Image(p))

## 7. Peek at the per-epoch numbers

In [ ]:
import pandas as pd, glob
t = sorted(glob.glob(AN + '/deep_joint_*_test.csv'))[-1]
tdf = pd.read_csv(t)
print('TEST RESULT PER SHUFFLE:'); display(tdf)
print('mean test AUC = %.4f +/- %.4f' % (tdf.test_auc.mean(), tdf.test_auc.std(ddof=0)))
e = sorted(glob.glob(AN + '/deep_joint_*_epochs.csv'))[-1]
edf = pd.read_csv(e)
print('per-epoch rows:', len(edf), '| shuffles:', edf.seed.nunique())
display(edf.tail())

## Want to go deeper / longer?
Change the settings in **cell 1** and re-run cells 1 and 5, e.g.:
- Deeper: `ENC_DEPTH = 8`, `ENC_WIDTH = 768`, `N_BLOCKS = 12`
- Longer/slower: `EPOCHS = 1000`, `LR = 5e-5`
- Compare heads: `HEAD = "mlp"`

**Note:** re-running overwrites the previous figure/CSV. To keep both, change
`DRIVE_BASE` (e.g. add `/run2`) before re-running.